# 自动驾驶感知教学实验 02：BEVDepth 2D 到 3D 视图变换与深度监督

> **教材定位**：本 Notebook 是《自动驾驶 BEV / Occupancy 感知教材》的第 02 实验单元。
> **真实性等级**：`[REAL MODEL OUTPUT & HOOK CACHE]` + `[SHAPE DEMO / DIMENSION SPECIFICATION]`
> **教学重点**：
> 1. 理解 LSS 视锥外积构建原理：$F_{3D}(u, v, d) = P(d|u,v) \otimes F_{2D}(u,v)$。
> 2. 区分技术机理：BEVDet 通过 3D 检测 Loss 隐式学习深度；BEVDepth 引入激光点云投影真实深度 (LiDAR Depth GT) 进行显式监督。
> 3. 读取本地模型 Hook 导出的真实 DepthNet 离散深度概率分布与 Voxel Pooling 特征流。


### [SHAPE DEMO / DIMENSION SPECIFICATION]
以下 Cell 演示 BEVDepth 7 阶段特征维度的数学流转逻辑（张量数值为教学 Shape Demo，用于理解维度变换）：


In [1]:
import torch
print("[SHAPE DEMO] BEVDepth 7 阶段特征张量维度流转：")

# 阶段 1: 环视图像输入
img = torch.zeros(1, 6, 3, 256, 704)
print(f"Stage 1 [Raw Input]:            {list(img.shape)} (B, N_cam, C, H, W)")

# 阶段 2: 图像主干与 FPN 特征
feat_c2 = torch.zeros(1, 6, 512, 16, 44)
print(f"Stage 2 [FPN C2 Feat]:          {list(feat_c2.shape)} (下采样倍率 16x)")

# 阶段 3: DepthNet 离散深度分布
depth_logits = torch.zeros(1, 6, 118, 16, 44) # D=118 区间 (1.0m ~ 60.0m)
print(f"Stage 3 [DepthNet P(d)]:        {list(depth_logits.shape)} (D=118 离散深度 Bin)")

# 阶段 4: 视锥特征体构建 (Lift 外积)
context = torch.zeros(1, 6, 80, 16, 44)
frustum_feat = torch.zeros(1, 6, 118, 80, 16, 44)
print(f"Stage 4 [Frustum Volume]:       {list(frustum_feat.shape)} (外积展开)")

# 阶段 5: 体素池化到 BEV
bev_pool = torch.zeros(1, 80, 128, 128)
print(f"Stage 5 [Voxel Pooling BEV]:    {list(bev_pool.shape)} (0.8m x 0.8m 分辨率)")

# 阶段 6: BEV 编码主干
bev_encoded = torch.zeros(1, 256, 128, 128)
print(f"Stage 6 [BEV Backbone Output]:  {list(bev_encoded.shape)}")

# 阶段 7: CenterHead 预测头
heatmap = torch.zeros(1, 10, 128, 128)
reg = torch.zeros(1, 2, 128, 128)
dim = torch.zeros(1, 3, 128, 128)
print(f"Stage 7 [CenterHead Heads]:     热图={list(heatmap.shape)}, 尺寸={list(dim.shape)}")


[SHAPE DEMO] BEVDepth 7 阶段特征张量维度流转：
Stage 1 [Raw Input]:            [1, 6, 3, 256, 704] (B, N_cam, C, H, W)
Stage 2 [FPN C2 Feat]:          [1, 6, 512, 16, 44] (下采样倍率 16x)
Stage 3 [DepthNet P(d)]:        [1, 6, 118, 16, 44] (D=118 离散深度 Bin)
Stage 4 [Frustum Volume]:       [1, 6, 118, 80, 16, 44] (外积展开)
Stage 5 [Voxel Pooling BEV]:    [1, 80, 128, 128] (0.8m x 0.8m 分辨率)
Stage 6 [BEV Backbone Output]:  [1, 256, 128, 128]
Stage 7 [CenterHead Heads]:     热图=[1, 10, 128, 128], 尺寸=[1, 3, 128, 128]


### [REAL MODEL OUTPUT & EXPERIMENT HOOK]
以下数据直接读取实验导出的真实 BEVDepth DepthNet 预测分布 `artifacts/experiment_exports/bevdepth/real_depth_ray.json`：


In [1]:
import json
export_path = '/public/lyh/projects/BEV/artifacts/experiment_exports/bevdepth/real_depth_ray.json'
with open(export_path) as f:
    ray_data = json.load(f)

meta = ray_data['metadata']
dist = ray_data['ray_distribution']

print(f"[REAL MODEL HOOK DATA]")
print(f"模型来源:   {meta['model']} ({meta['checkpoint']})")
print(f"查询像素:   u={meta['pixel_query'][0]}, v={meta['pixel_query'][1]}")
print(f"预测峰值深度: {meta['peak_depth_m']} 米")

# 打印最高概率的前 3 个 Depth Bins
sorted_bins = sorted(dist, key=lambda x: x['prob'], reverse=True)[:3]
print("\nTop 3 深度置信度区间:")
for b in sorted_bins:
    print(f"  Bin #{b['bin_idx']:2d}: 距离 {b['depth_m']:.2f}m | 概率 P(d) = {b['prob']*100:.2f}% | Ego 坐标 = {b['ego_coord']}")


[REAL MODEL HOOK DATA]
模型来源:   BEVDepth-R50 (/public/lyh/pretrained_model/BEV/bevdet/bevdet-r50-4d-depth-cbgs.pth)
查询像素:   u=800.0, v=490.0
预测峰值深度: 14.5 米

Top 3 深度置信度区间:
  Bin #26: 距离 14.25m | 概率 P(d) = 8.47% | Ego 坐标 = [1.52, 14.27, 1.53]
  Bin #27: 距离 14.75m | 概率 P(d) = 8.47% | Ego 坐标 = [1.51, 14.77, 1.53]
  Bin #25: 距离 13.75m | 概率 P(d) = 7.12% | Ego 坐标 = [1.52, 13.77, 1.53]
